# Tokenizers

## Into to tokenization

Raw text is generally represented as Unicode strings.

**Unicode** is a standard that defines a character set with unique numbers (code points) for every character, while **UTF-8** is a specific way of encoding those numbers into bytes for storage and transmission.

In UTF-8 each character is encoded as 1 to 4 bytes.

In [1]:
string = "Привіт світ! Hello, 🌍!"

A language model places a probability distribution over sequences of tokens (usually represented by integer indices).

In [2]:
indices = [23881, 3820, 24127, 159109, 0, 32949, 11, 130321, 235, 0]

So we need a procedure that encodes strings into tokens.

We also need a procedure that decodes tokens back into strings.

A  **Tokenizer** is a class that implements the encode and decode methods.

The vocabulary size is number of possible tokens (integers).

In [3]:
from abc import ABC

class Tokenizer(ABC):
  """Abstract interface for a tokenizer."""
  def encode(self, string: str) -> list[int]:
      raise NotImplementedError

  def decode(self, indices: list[int]) -> str:
      raise NotImplementedError

To feel how tokenizers work, [play](https://tiktokenizer.vercel.app/?encoder=gpt2)

#### Key questions:

* How space is tokenized?
* Are the same word tokenizes differently depending on its place in the sentence?
* How numbers are tokenized?

In [4]:
import tiktoken

def get_compression_ratio(string: str, indices: list[int]) -> float:
    num_bytes = len(bytes(string, encoding="utf-8"))
    num_tokens = len(indices)
    return num_bytes / num_tokens


gpt2_tokenizer = tiktoken.get_encoding("gpt2")

indices = gpt2_tokenizer.encode(string)
reconstructed_string = gpt2_tokenizer.decode(indices)
assert string == reconstructed_string
get_compression_ratio(string, indices)

1.6666666666666667

## Simplest tokenizer

In [5]:
class CharacterTokenizer(Tokenizer):
  """Represent a string as a sequence of Unicode code points."""
  def encode(self, string: str) -> list[int]:
      return list(map(ord, string))

  def decode(self, indices: list[int]) -> str:
      return "".join(map(chr, indices))

In [6]:
char_tokenizer = CharacterTokenizer()
char_tokenizer.encode(string)

[1055,
 1088,
 1080,
 1074,
 1110,
 1090,
 32,
 1089,
 1074,
 1110,
 1090,
 33,
 32,
 72,
 101,
 108,
 108,
 111,
 44,
 32,
 127757,
 33]

In [7]:
char_tokenizer = CharacterTokenizer()
indices = char_tokenizer.encode(string)
reconstructed_string = char_tokenizer.decode(indices)
assert string == reconstructed_string
get_compression_ratio(string, indices)

1.5909090909090908

#### Any problems?

1. Very large vocabulary.
2. Inefficient use of it. Many characters are rare (e.g., 🌍).


## Byte-based tokenizer

Unicode strings can be represented as a sequence of bytes, which can be represented by integers between 0 and 255.

There are some tokenizer-free approaches, which work on raw bytes e.g. [paper 1](https://arxiv.org/abs/2105.13626), [paper 2](https://arxiv.org/abs/2412.09871) and more.

In [8]:
# some Unicode characters are represented by one byte
bytes("a", encoding="utf-8")

b'a'

In [9]:
 # others take multiple bytes
bytes("🌍", encoding="utf-8")

b'\xf0\x9f\x8c\x8d'

In [10]:
bytes("і", encoding="utf-8")

b'\xd1\x96'

In [11]:
class ByteTokenizer(Tokenizer):
  def encode(self, string: str) -> list[int]:
    string_bytes = string.encode("utf-8")
    indices = list(map(int, string_bytes))
    return indices

  def decode(self, indices: list[int]) -> str:
    string_bytes = bytes(indices)
    string = string_bytes.decode("utf-8")
    return string

In [12]:
byte_tokenizer = ByteTokenizer()
indices = byte_tokenizer.encode(string)
reconstructed_string = byte_tokenizer.decode(indices)
assert string == reconstructed_string

#### What about the compression rate?

In [13]:
get_compression_ratio(string, indices)

1.0

There is no compression. Which means the sequences will be too long. Recap that the attention is quadratic, so this is not looking great.

## Word-based tokenizer


Classical approach. Just split sequences into words.

In [17]:
import regex

class WordTokenizer(Tokenizer):
  def __init__(self, vocab: dict[int, str], unk_idx: int = -1):
    self.unk_idx = unk_idx
    self.vocab = vocab
    self.reverse_vocab = {v:k for k,v in vocab.items()}

  def encode(self, string: str) -> list[int]:
    words = regex.findall(r"\w+|.", string)
    indices = [self.reverse_vocab.get(word, self.unk_idx) for word in words]
    return indices

  def decode(self, indices: list[int]) -> str:
    return "".join(map(self.vocab.get, indices))



def train_word_tokenizer(strings: list[str]):
  words = []
  for string in strings:
    words.extend(regex.findall(r"\w+|.", string))

  unique_words = sorted(list(set(words)))
  vocab = {i: word for i, word in enumerate(unique_words)}

  unk_idx = len(vocab)
  vocab[unk_idx] = "<unk>"

  return WordTokenizer(vocab, unk_idx)

In [18]:
word_tokenizer = train_word_tokenizer([string])
indices = word_tokenizer.encode(string)
reconstructed_string = word_tokenizer.decode(indices)
assert string == reconstructed_string

In [19]:
word_tokenizer.encode("Дніпро реве")

[7, 0, 7]

In [20]:
word_tokenizer.decode([7, 0, 7])

'<unk> <unk>'

#### Why not?

1. The vocabulary size is huge. Doesn't provide a fixed vocabulary size.
2. Many words are rare, the model won't learn much about them
3. New words, which haven't seen during training, get a special UNK token. This could ruin everything.


## General tokenization algorithm

1. Normalization: some general cleanup, such as removing needless whitespace, lowercasing, and/or removing accents.
2. Pre-tokenization: splitting text into words.
3. Model: splitting into tokens.
4. Postprocessor: adding special tokens etc.

**Normalization**

In [22]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer.backend_tokenizer.normalizer.normalize_str("Héllò hôw are ü?")

'hello how are u?'

**Pre-tokenization**

In [23]:
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?")

[('Hello', (0, 5)),
 (',', (5, 6)),
 ('how', (7, 10)),
 ('are', (11, 14)),
 ('you', (16, 19)),
 ('?', (19, 20))]

In [25]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?")

[('Hello', (0, 5)),
 (',', (5, 6)),
 ('Ġhow', (6, 10)),
 ('Ġare', (10, 14)),
 ('Ġ', (14, 15)),
 ('Ġyou', (15, 19)),
 ('?', (19, 20))]

In [27]:
tokenizer = AutoTokenizer.from_pretrained("t5-small")
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?")

[('▁Hello,', (0, 6)),
 ('▁how', (7, 10)),
 ('▁are', (11, 14)),
 ('▁you?', (16, 20))]

#### Q: Do we always follow this pipeline?


## The **model** example

The model is trained on the training corpora like this:
1. Build the base vocabulary by taking all the symbols used to write those words
2. Learn to merge two tokens into one by searching for most frequent two consecutive tokens in a word.

Then on inference after normalization and pre-tokenization lets:

1. Split the words into individual characters
2. Apply the merge rules learned in order on those splits

In [28]:
corpus = [
    "This is the NLP Course.",
    "Today we talk about tokenization.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [29]:
from collections import defaultdict

tokenizer = AutoTokenizer.from_pretrained("gpt2")

def compute_pair_freqs(splits, word_freqs):
  pair_freqs = defaultdict(int)
  for word, freq in word_freqs.items():
    split = splits[word]
    if len(split) == 1:
      continue
    for i in range(len(split) - 1):
      pair = (split[i], split[i + 1])
      pair_freqs[pair] += freq
  return pair_freqs

def merge_pair(a, b, splits, word_freqs):
  for word in word_freqs:
    split = splits[word]
    if len(split) == 1:
      continue

    i = 0
    while i < len(split) - 1:
      if split[i] == a and split[i + 1] == b:
        split = split[:i] + [a + b] + split[i + 2 :]
      else:
        i += 1
    splits[word] = split
  return splits

In [30]:
def train_the_model(corpus, vocab_size: int):
  word_freqs = defaultdict(int)

  for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
      word_freqs[word] += 1

  vocab = []

  for word in word_freqs.keys():
    for letter in word:
      if letter not in vocab:
        vocab.append(letter)
  vocab.sort()

  splits = {word: [c for c in word] for word in word_freqs.keys()}

  merges = {}

  while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits, word_freqs)
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    splits = merge_pair(*best_pair, splits, word_freqs)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

  return vocab, merges

vocab, merges = train_the_model(corpus, 50)

In [31]:
print(vocab)

[',', '.', 'C', 'H', 'L', 'N', 'P', 'T', 'a', 'b', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'n', 'o', 'p', 'r', 's', 't', 'u', 'w', 'y', 'z', 'Ġ', 'Ġt', 'Ġa', 'ou', 'Ġto', 'en', 'nd', 'is', 'Ġth', 'Ġthe', 'rs', 'Ġw', 'Ġab', 'Ġtok', 'Ġtoken', 'at', 'll', 'Th', 'This', 'Ġis', 'ĠN', 'ĠNL']


In [32]:
merges

{('Ġ', 't'): 'Ġt',
 ('Ġ', 'a'): 'Ġa',
 ('o', 'u'): 'ou',
 ('Ġt', 'o'): 'Ġto',
 ('e', 'n'): 'en',
 ('n', 'd'): 'nd',
 ('i', 's'): 'is',
 ('Ġt', 'h'): 'Ġth',
 ('Ġth', 'e'): 'Ġthe',
 ('r', 's'): 'rs',
 ('Ġ', 'w'): 'Ġw',
 ('Ġa', 'b'): 'Ġab',
 ('Ġto', 'k'): 'Ġtok',
 ('Ġtok', 'en'): 'Ġtoken',
 ('a', 't'): 'at',
 ('l', 'l'): 'll',
 ('T', 'h'): 'Th',
 ('Th', 'is'): 'This',
 ('Ġ', 'is'): 'Ġis',
 ('Ġ', 'N'): 'ĠN',
 ('ĠN', 'L'): 'ĠNL'}

In [33]:
def tokenize(text, merges):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    splits = [[l for l in word] for word in pre_tokenized_text]
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])

In [34]:
print(tokenize("This is not a token.", merges))

['This', 'Ġis', 'Ġ', 'n', 'o', 't', 'Ġa', 'Ġtoken', '.']


#### Q: What is missing?

## Byte Pair Encoding (BPE) tokenizer

[Introduced](http://www.pennelynn.com/Documents/CUJ/HTML/94HTML/19940045.HTM) in 1994 for data compression.

[Adapted](https://arxiv.org/abs/1508.07909) to machine translation in 2015.

Then was used by [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf).

Now BPE tokenizers are widely used in large language models due to their balance of vocabulary compactness and representational power.

Basic Idea:
* common sequences of characters are represented by a single token, rare sequences are represented by many tokens;
* start with each byte as a token, and successively merge the most common pair of adjacent tokens;

In [35]:
def merge(indices: list[int], pair: tuple[int, int], new_index: int) -> list[int]:
  new_indices = []
  i = 0
  while i < len(indices):
      if i + 1 < len(indices) and indices[i] == pair[0] and indices[i + 1] == pair[1]:
          new_indices.append(new_index)
          i += 2
      else:
          new_indices.append(indices[i])
          i += 1
  return new_indices

In [36]:
from collections import defaultdict


class BPETokenizer(Tokenizer):
  def __init__(self, vocab: dict[int, bytes], merges: dict[tuple[int, int], int]):
      self.vocab = vocab
      self.merges = merges

  def encode(self, string: str) -> list[int]:
      indices = list(map(int, string.encode("utf-8")))
      # it can be better
      for pair, new_index in self.merges.items():
          indices = merge(indices, pair, new_index)
      return indices

  def decode(self, indices: list[int]) -> str:
      bytes_list = list(map(self.vocab.get, indices))
      string = b"".join(bytes_list).decode("utf-8")
      return string



def train_bpe(corpus: list[str], num_merges: int) -> BPETokenizer:
    batch = [[int(i) for i in string.encode("utf-8")] for string in corpus]
    merges: dict[tuple[int, int], int] = {}
    vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
    for i in range(num_merges):
        counts = defaultdict(int)
        for indices in batch:
          for index1, index2 in zip(indices, indices[1:]):
              counts[(index1, index2)] += 1
        pair = max(counts, key=counts.get)
        index1, index2 = pair
        new_index = 256 + i
        merges[pair] = new_index
        vocab[new_index] = vocab[index1] + vocab[index2]
        batch = [merge(indices, pair, new_index) for indices in batch]
    return BPETokenizer(vocab=vocab, merges=merges)

In [37]:
bpe_tokenizer = train_bpe(corpus, num_merges=30)

def test_bpe(string):
  indices = bpe_tokenizer.encode(string)
  print(indices)
  reconstructed_string = bpe_tokenizer.decode(indices)
  print(reconstructed_string)
  assert string == reconstructed_string

In [38]:
test_bpe(string)

[208, 159, 209, 128, 208, 184, 208, 178, 209, 150, 209, 130, 32, 209, 129, 208, 178, 209, 150, 209, 130, 33, 32, 72, 101, 271, 111, 44, 32, 240, 159, 140, 141, 33]
Привіт світ! Hello, 🌍!


In [39]:
test_bpe("Реве та стогне Дніпро широкий")

[208, 160, 208, 181, 208, 178, 208, 181, 32, 209, 130, 208, 176, 32, 209, 129, 209, 130, 208, 190, 208, 179, 208, 189, 208, 181, 32, 208, 148, 208, 189, 209, 150, 208, 191, 209, 128, 208, 190, 32, 209, 136, 208, 184, 209, 128, 208, 190, 208, 186, 208, 184, 208, 185]
Реве та стогне Дніпро широкий


In [40]:
test_bpe("Hello tokens!")

[72, 101, 271, 111, 256, 269, 115, 33]
Hello tokens!


The GPT-2 paper used word-based tokenization to break up the text into inital segments and run the original BPE algorithm on each segment.

## Materials

1. Stanford's CS336 [Language Modeling from Scratch](https://stanford-cs336.github.io/spring2025/)
2. HuggingFace's [LLM Course](https://huggingface.co/learn/llm-course)
3. Andrej Karpathy's [video on tokenization](https://www.youtube.com/watch?v=zduSFxRajkE)